# 🎬 開源 AI 影片生成工作流 (Google Colab 免費 GPU 專用)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MazingerSam/video_test/blob/main/notebooks/colab_ai_video_workflow.ipynb)

本筆記本 **100% 免費、無須任何第三方 API 金鑰**，直接利用 Google Colab 提供的免費 GPU (T4) 運行兩大主流開源 AI 影片模型：
1. **AnimateDiff (文字轉影片 / Text-to-Video)**：輸入英文提示詞，自動產出流暢動態短影片。
2. **Stable Video Diffusion - SVD (圖片轉影片 / Image-to-Video)**：上傳任意靜態圖片，自動生成電影感運鏡動態影片。

## 步驟 0：檢查 GPU 環境
> **重要提示**：請確認已開啟 GPU 加速器！
> 在上方選單點選 **「執行階段 (Runtime)」➔「變更執行階段類型 (Change runtime type)」➔ 選擇「T4 GPU」**。

In [ ]:
# 檢查顯卡狀態
!nvidia-smi

## 步驟 1：安裝開源 AI 核心套件
安裝 Hugging Face `diffusers`, `transformers`, `accelerate` 與影片輸出工具。

In [ ]:
# 安裝 diffusers, transformers, accelerate, imageio-ffmpeg
!pip install -q -U diffusers transformers accelerate imageio[ffmpeg] safetensors

## 步驟 2：工作流 1 — AnimateDiff (文字轉影片 / Text-to-Video)
輸入提示詞，直接由文字生成影片。

In [ ]:
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
from diffusers.utils import export_to_video
from IPython.display import HTML
from pathlib import Path
import time

OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("⏳ 正在載入 AnimateDiff 運動適配器與基礎模型 (首次需下載權重約 2~3 分鐘)... ")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# 載入 Motion Adapter 與 Realistic Vision 模型
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=dtype)
pipe_t2v = AnimateDiffPipeline.from_pretrained(
    "SG161222/Realistic_Vision_V5.1_noVAE",
    motion_adapter=adapter,
    torch_dtype=dtype
)
pipe_t2v.scheduler = DDIMScheduler.from_config(
    pipe_t2v.scheduler.config,
    beta_schedule="linear",
    clip_sample=False,
    timestep_spacing="linspace",
    steps_offset=1
)

# 啟用顯存優化 (防 T4 OOM 記憶體溢出)
pipe_t2v.enable_vae_slicing()
pipe_t2v.enable_model_cpu_offload()
print("✅ AnimateDiff 模型載入完成！")

In [ ]:
# 設定提示詞並生成文字影片
prompt = "masterpiece, best quality, a futuristic astronaut walking on alien planet, glowing crystal flowers, cinematic lighting, photorealistic, 4k"
negative_prompt = "bad quality, worse quality, blurry, distorted, deformed, watermark, lowres"

print("🎬 正在生成文字影片 (約需 1~2 分鐘)... ")
output = pipe_t2v(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_frames=16,
    guidance_scale=7.5,
    num_inference_steps=25,
    generator=torch.Generator(device=device).manual_seed(42)
)
frames = output.frames[0]

t2v_output_file = OUTPUT_DIR / f"animatediff_{int(time.time())}.mp4"
export_to_video(frames, str(t2v_output_file), fps=8)
print(f"🎉 文字影片生成完成！已儲存至: {t2v_output_file}")

# 於 Colab 內播放影片
HTML(f'''
<video width="512" height="512" controls autoplay loop>
  <source src="{t2v_output_file}" type="video/mp4">
</video>
''')

## 步驟 3：工作流 2 — Stable Video Diffusion (圖片轉影片 / Image-to-Video)
給定一張靜態圖片，利用 Stability AI 的 SVD-XT 模型生成流暢的動態運鏡。

In [ ]:
from diffusers import StableVideoDiffusionPipeline
from PIL import Image
import requests
from io import BytesIO

print("⏳ 正在載入 Stable Video Diffusion XT (SVD-XT) 模型... ")
# 載入 SVD Pipeline
pipe_i2v = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16"
)

# 啟用顯存優化
pipe_i2v.enable_vae_slicing()
pipe_i2v.enable_model_cpu_offload()
print("✅ SVD-XT 模型載入完成！")

In [ ]:
# 準備輸入圖片 (可填入網路圖片 URL，或上傳圖片後指定本地檔名)
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/svd/rocket.png"

response = requests.get(image_url)
input_image = Image.open(BytesIO(response.content)).convert("RGB")
input_image = input_image.resize((1024, 576))

display(input_image)
print("📷 輸入圖片已就緒，準備生成影片...")

In [ ]:
# 生成圖片轉影片
print("🎬 正在執行 SVD 運鏡生成 (約需 2~3 分鐘)... ")
generator = torch.manual_seed(42)
frames = pipe_i2v(
    input_image,
    decode_chunk_size=8,
    generator=generator,
    motion_bucket_id=127,
    num_frames=25
).frames[0]

svd_output_file = OUTPUT_DIR / f"svd_{int(time.time())}.mp4"
export_to_video(frames, str(svd_output_file), fps=7)
print(f"🎉 圖片影片生成完成！已儲存至: {svd_output_file}")

# 於 Colab 內播放影片
HTML(f'''
<video width="640" height="360" controls autoplay loop>
  <source src="{svd_output_file}" type="video/mp4">
</video>
''')

## 步驟 4：(可選) 掛載 Google Drive 同步保存
一鍵將在 Colab 產出的 MP4 影片備份至您的 Google 雲端硬碟。

In [ ]:
from google.colab import drive
import shutil

# 掛載雲端硬碟
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/Colab_AI_Videos')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

for mp4 in OUTPUT_DIR.glob("*.mp4"):
    dest = DRIVE_DIR / mp4.name
    shutil.copyfile(mp4, dest)
    print(f"☁️ 已同步備份: {dest}")

print("✅ 所有影片已成功儲存至 Google Drive！")